# E4: steering with vs without the preamble (HF Jobs L4, bf16)

In [ ]:
!git clone -q -b feat/hsafe https://github.com/dhairya-pandya/Lexsi-AI-Alignment repo
%cd repo
!pip install -q -e '.[gpu]'

In [ ]:
import json, yaml, numpy as np, pandas as pd, torch, os
from hsafe.interp.hooks import load_model
from hsafe.interp.directions import refusal_direction
from hsafe.interp.steer import steer_sweep, random_direction_like, unembed_cosine
from hsafe.records import read_jsonl, PromptRecord
from hsafe.render import render
m = yaml.safe_load(open('configs/models.yaml'))['tiny-aya-global']
model, tok = load_model(m['id'], m['revision'], torch.bfloat16)
openers = json.load(open('results/gate_g1/openers.json'))
H = read_jsonl('data/prompts/harmful_en.jsonl', PromptRecord); B = read_jsonl('data/prompts/benign.jsonl', PromptRecord)
fit_h = [r for r in H if r.split=='fit' and r.bucket!='ACCEPT'][:40]; fit_b = [r for r in B if r.split=='fit'][:40]
ev_h = [r for r in H if r.split=='eval' and r.bucket!='ACCEPT'][:60]; ev_b = [r for r in B if r.split=='eval'][:60]
os.makedirs('results/e4', exist_ok=True)

In [ ]:
# layer sweep on fit: pick the layer whose direction best separates harmful vs benign (cross-check with SafeTune)
L = 24
R = lambda recs, c: [render(r, c, None, None, tok, 'tiny_aya') for r in recs]
dirs = {c: refusal_direction(model, R(fit_h, c), R(fit_b, c), L) for c in ['P-default', 'P-none']}
for c, d in dirs.items(): torch.save(d, f'results/e4/direction_{c}_L{L}.pt')

In [ ]:
coeffs = [-2, -1, -0.5, 0, 0.5, 1, 2]
frames = []
for cond in ['P-default', 'P-none']:
    for kind, recs in [('harmful', ev_h), ('benign', ev_b)]:
        rs = R(recs, cond)
        for dname, d in [('true', dirs[cond])] + [(f'random{s}', random_direction_like(dirs[cond], s)) for s in range(5)]:
            df = steer_sweep(model, rs, d, L, coeffs, openers); df['kind'] = kind; df['direction'] = dname
            frames.append(df)
sweep = pd.concat(frames); sweep.to_csv('results/e4/sweep.csv', index=False)
curve = sweep.groupby(['condition','kind','direction','coeff']).p_refusal.mean().reset_index()
base = curve[curve.coeff==0].set_index(['condition','kind','direction']).p_refusal
curve['delta'] = curve.apply(lambda r: r.p_refusal - base[(r.condition, r.kind, r.direction)], axis=1)
auc = curve.groupby(['condition','kind','direction']).apply(lambda g: np.trapezoid(g.sort_values('coeff').delta, g.sort_values('coeff').coeff))
auc.to_csv('results/e4/auc.csv'); auc

In [ ]:
cos = unembed_cosine(model, dirs['P-default'], openers)
pd.DataFrame({'token_id': openers, 'token': [tok.decode([t]) for t in openers], 'cosine': cos}).to_csv('results/e4/unembed_cosine.csv', index=False)